# Canopy memory in a durian orchard: a dual-timescale light-use-efficiency chain
**Weather (ERA5) → stress scalars → GPP → leaf pool *L* (fast) + crown cover *c* (slow) → NDVI (Sentinel-2)**

Orchard polygon: Chanthaburi, Thailand (centroid 12.6510 °N, 102.2128 °E, ≈ 7.2 ha). Period: 1 Jan 2020 – 24 Sep 2026.

Steps in this notebook
1. Download hourly ERA5-Land / ERA5 for the orchard (Open-Meteo archive API) and aggregate to daily drivers.
2. Load the Sentinel-2 L2A indices (Cloud Score+ and SCL masked; extracted with Google Earth Engine).
3. Soil hydraulic thresholds from SoilGrids 2.0 (root zone 0–100 cm).
4. Model: f(T), f(VPD), f(SM), GPP, dual-timescale canopy, NDVI.
5. Calibration with literature values fixed, leave-one-year-out comparison and figures.

All fixed parameter values come from the literature (MOD17 BPLUT evergreen broadleaf forest, FAO-56, SoilGrids). Only canopy parameters are estimated.

In [ ]:
import numpy as np, pandas as pd, io, json, requests, warnings, time
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, differential_evolution
warnings.filterwarnings('ignore')
try:
    import numba
except ImportError:
    !pip -q install numba
print('ready')

## 1. ERA5 drivers (Open-Meteo archive API)

In [ ]:
LAT, LON = 12.651007, 102.212826
START, END = '2020-01-01', '2026-09-24'
BASE = 'https://archive-api.open-meteo.com/v1/archive'
vL = ('temperature_2m,dew_point_2m,relative_humidity_2m,vapour_pressure_deficit,'
      'soil_temperature_0_to_7cm,soil_temperature_7_to_28cm,soil_temperature_28_to_100cm,soil_temperature_100_to_255cm,'
      'soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm,soil_moisture_100_to_255cm')
vA = ('precipitation,surface_pressure,cloud_cover,shortwave_radiation,direct_radiation,diffuse_radiation,'
      'wind_speed_10m,wind_direction_10m,et0_fao_evapotranspiration')
def get(hourly, model):
    q = dict(latitude=LAT, longitude=LON, start_date=START, end_date=END, hourly=hourly, models=model,
             timezone='Asia/Bangkok', wind_speed_unit='ms')
    for attempt in range(5):
        r = requests.get(BASE, params=q, timeout=120)
        if r.status_code == 200: return r.json()
        print('retry', r.status_code, r.text[:120]); time.sleep(65)
    raise RuntimeError('Open-Meteo request failed')
L = get(vL, 'era5_land'); time.sleep(5); A = get(vA, 'era5')
h = pd.DataFrame(L['hourly']).merge(pd.DataFrame(A['hourly']), on='time')
h['time'] = pd.to_datetime(h['time']); h = h.set_index('time')
h['PAR_MJ_m2_h'] = h['shortwave_radiation'] * 0.0036 * 0.48      # W m-2 -> MJ m-2 h-1, PAR fraction 0.48
print('ERA5-Land grid', L['latitude'], L['longitude'], '| ERA5 grid', A['latitude'], A['longitude'], '| hours', len(h))
day = h['shortwave_radiation'] > 0
g = h.resample('D')
d = pd.DataFrame({
    'T_daytime_C': h.loc[day, 'temperature_2m'].resample('D').mean(),
    'VPD_daytime_kPa': h.loc[day, 'vapour_pressure_deficit'].resample('D').mean(),
    'PAR_MJ_m2_d': g['PAR_MJ_m2_h'].sum(),
    'P_mm_d': g['precipitation'].sum(),
    'ET0_mm_d': g['et0_fao_evapotranspiration'].sum(),
})
d['SM_rootzone_0_100cm'] = (7*g['soil_moisture_0_to_7cm'].mean() + 21*g['soil_moisture_7_to_28cm'].mean()
                            + 72*g['soil_moisture_28_to_100cm'].mean()) / 100
d.describe().T.round(2)

## 2. Sentinel-2 L2A indices for the orchard polygon
Extracted in Google Earth Engine (`COPERNICUS/S2_SR_HARMONIZED`, Cloud Score+ `cs_cdf ≥ 0.6`, SCL classes 3, 8, 9, 10, 11 removed). The Earth Engine script is given at the end of the notebook. Only scenes with ≥ 90 % clear pixels are used for calibration.

In [ ]:
S2_CSV = '''date,scene_cloud_pct,n_clear,NDVI,EVI,NDRE,NDMI,n_pix,clear_frac,quality
2020-01-04,3.0,745.0,0.4956,0.3278,0.317,-0.0803,745.0,1.0,good(>=90%)
2020-01-09,2.0,745.0,0.4642,0.3022,0.2802,-0.1051,745.0,1.0,good(>=90%)
2020-01-14,9.0,745.0,0.4553,0.3279,0.3004,-0.0804,745.0,1.0,good(>=90%)
2020-01-19,25.0,745.0,0.4598,0.3463,0.2942,-0.0891,745.0,1.0,good(>=90%)
2020-01-29,20.0,745.0,0.4692,0.3504,0.2975,-0.0683,745.0,1.0,good(>=90%)
2020-02-08,5.0,745.0,0.4609,0.322,0.2951,-0.0708,745.0,1.0,good(>=90%)
2020-02-23,0.0,745.0,0.455,0.3472,0.2923,-0.0617,745.0,1.0,good(>=90%)
2020-03-09,2.0,745.0,0.4317,0.3211,0.2713,-0.1264,745.0,1.0,good(>=90%)
2020-03-14,25.0,260.0,0.3496,0.2956,0.2159,-0.1209,745.0,0.349,poor(<50%)
2020-03-29,12.0,29.0,0.5089,0.5688,0.3595,0.1089,745.0,0.039,poor(<50%)
2020-04-03,30.0,89.0,0.3798,0.5482,0.2522,-0.0223,745.0,0.119,poor(<50%)
2020-04-08,8.0,493.0,0.4843,0.3522,0.3088,0.0068,745.0,0.662,partial
2020-04-18,36.0,745.0,0.4965,0.3772,0.2979,-0.0668,745.0,1.0,good(>=90%)
2020-04-28,29.0,709.0,0.4499,0.514,0.2918,0.1943,745.0,0.952,good(>=90%)
2020-06-12,24.0,528.0,0.6082,0.5112,0.3879,0.0772,745.0,0.709,partial
2020-07-07,34.0,737.0,0.758,0.8293,0.4855,0.2512,745.0,0.989,good(>=90%)
2020-07-22,63.0,745.0,0.7546,0.748,0.4899,0.2401,745.0,1.0,good(>=90%)
2020-08-26,25.0,729.0,0.434,0.3004,0.2551,-0.0657,745.0,0.979,good(>=90%)
2020-09-05,27.0,741.0,0.4813,0.4391,0.2819,-0.0443,745.0,0.995,good(>=90%)
2020-09-10,31.0,16.0,0.5893,0.5674,0.4082,0.1607,745.0,0.021,poor(<50%)
2020-09-25,12.0,266.0,0.5344,0.243,0.294,-0.0108,745.0,0.357,poor(<50%)
2020-10-20,74.0,745.0,0.6234,0.5188,0.3799,0.0759,745.0,1.0,good(>=90%)
2020-11-09,18.0,745.0,0.5247,0.3909,0.3314,-0.0231,745.0,1.0,good(>=90%)
2020-11-14,41.0,349.0,0.4967,0.3581,0.3244,-0.0208,745.0,0.468,poor(<50%)
2020-11-19,4.0,745.0,0.4878,0.3244,0.2986,-0.0961,745.0,1.0,good(>=90%)
2020-12-09,0.0,745.0,0.5819,0.4079,0.3802,0.0064,745.0,1.0,good(>=90%)
2020-12-19,20.0,745.0,0.5746,0.3993,0.3703,-0.0196,745.0,1.0,good(>=90%)
2021-01-03,5.0,745.0,0.5425,0.3601,0.3498,-0.0238,745.0,1.0,good(>=90%)
2021-01-08,45.0,745.0,0.5463,0.4314,0.3729,0.0011,745.0,1.0,good(>=90%)
2021-01-13,0.0,745.0,0.5387,0.3758,0.3568,-0.0158,745.0,1.0,good(>=90%)
2021-01-18,24.0,745.0,0.4909,0.3483,0.3324,-0.0294,745.0,1.0,good(>=90%)
2021-01-23,1.0,745.0,0.4776,0.3156,0.3027,-0.094,745.0,1.0,good(>=90%)
2021-01-28,26.0,732.0,0.4172,0.2446,0.2737,0.0137,745.0,0.983,good(>=90%)
2021-02-02,17.0,745.0,0.4362,0.3028,0.2852,-0.086,745.0,1.0,good(>=90%)
2021-02-07,5.0,745.0,0.4402,0.3093,0.2899,-0.1078,745.0,1.0,good(>=90%)
2021-02-12,4.0,745.0,0.5114,0.356,0.3381,-0.0453,745.0,1.0,good(>=90%)
2021-02-17,33.0,19.0,0.4985,0.4113,0.3481,0.1786,745.0,0.026,poor(<50%)
2021-02-22,6.0,745.0,0.4874,0.3735,0.313,-0.0472,745.0,1.0,good(>=90%)
2021-02-27,0.0,745.0,0.4708,0.3303,0.3066,-0.1069,745.0,1.0,good(>=90%)
2021-03-04,48.0,424.0,0.455,0.3741,0.2869,-0.0604,745.0,0.569,partial
2021-03-14,40.0,125.0,0.3875,0.4515,0.265,-0.018,745.0,0.168,poor(<50%)
2021-03-24,15.0,745.0,0.5747,0.4762,0.3799,0.0438,745.0,1.0,good(>=90%)
2021-04-18,32.0,16.0,0.4699,0.3675,0.3183,0.0713,745.0,0.021,poor(<50%)
2021-05-08,24.0,745.0,0.628,0.4958,0.3855,0.0683,745.0,1.0,good(>=90%)
2021-05-13,33.0,741.0,0.5806,0.5571,0.36,0.0862,745.0,0.995,good(>=90%)
2021-05-23,43.0,745.0,0.6958,0.6327,0.4269,0.1388,745.0,1.0,good(>=90%)
2021-06-02,45.0,636.0,0.6373,0.6067,0.4062,0.1921,745.0,0.854,partial
2021-07-02,22.0,99.0,0.5648,0.5231,0.3549,0.1148,745.0,0.133,poor(<50%)
2021-08-21,39.0,745.0,0.6198,0.4994,0.3511,0.0855,745.0,1.0,good(>=90%)
2021-10-25,21.0,745.0,0.7234,0.5883,0.4688,0.1787,745.0,1.0,good(>=90%)
2021-11-04,60.0,745.0,0.7217,0.5996,0.4702,0.1757,745.0,1.0,good(>=90%)
2021-11-19,44.0,745.0,0.5816,0.4194,0.3445,0.0323,745.0,1.0,good(>=90%)
2021-11-24,69.0,43.0,0.4969,0.4953,0.3415,0.1052,745.0,0.058,poor(<50%)
2021-12-09,14.0,745.0,0.4912,0.3036,0.3108,-0.0996,745.0,1.0,good(>=90%)
2021-12-29,9.0,745.0,0.4661,0.2759,0.2949,-0.1444,745.0,1.0,good(>=90%)
2022-01-03,25.0,444.0,0.3871,0.4012,0.2704,0.0167,745.0,0.596,partial
2022-01-08,14.0,745.0,0.4508,0.284,0.2974,-0.1147,745.0,1.0,good(>=90%)
2022-01-13,2.0,745.0,0.4577,0.2866,0.3043,-0.1285,745.0,1.0,good(>=90%)
2022-01-18,7.0,356.0,0.4325,0.2077,0.2818,-0.0509,745.0,0.478,poor(<50%)
2022-01-28,19.0,710.0,0.411,0.3084,0.2599,-0.1506,745.0,0.953,good(>=90%)
2022-02-02,70.0,52.0,0.3638,0.426,0.242,-0.0443,745.0,0.07,poor(<50%)
2022-02-07,23.0,745.0,0.4189,0.2684,0.2721,-0.1375,745.0,1.0,good(>=90%)
2022-02-17,66.0,745.0,0.4801,0.329,0.293,-0.1022,745.0,1.0,good(>=90%)
2022-02-27,17.0,745.0,0.5411,0.4014,0.3472,-0.0179,745.0,1.0,good(>=90%)
2022-03-29,45.0,745.0,0.6251,0.5241,0.3752,0.0724,745.0,1.0,good(>=90%)
2022-04-03,52.0,745.0,0.512,0.3799,0.3099,-0.0621,745.0,1.0,good(>=90%)
2022-04-13,23.0,619.0,0.303,0.4778,0.2127,-0.0857,745.0,0.831,partial
2022-04-18,40.0,745.0,0.4383,0.321,0.2741,-0.0954,745.0,1.0,good(>=90%)
2022-04-28,52.0,745.0,0.4154,0.4118,0.2634,-0.0411,745.0,1.0,good(>=90%)
2022-07-27,80.0,745.0,0.6856,0.7004,0.4319,0.2031,745.0,1.0,good(>=90%)
2022-10-05,52.0,687.0,0.6757,0.5563,0.4174,0.1529,745.0,0.922,good(>=90%)
2022-10-25,74.0,705.0,0.6102,0.4567,0.3972,0.0993,745.0,0.946,good(>=90%)
2022-10-30,19.0,745.0,0.6712,0.4977,0.4413,0.1065,745.0,1.0,good(>=90%)
2022-11-04,1.0,745.0,0.646,0.4538,0.4223,0.0886,745.0,1.0,good(>=90%)
2022-11-14,86.0,32.0,0.5502,0.3778,0.3691,0.1127,745.0,0.043,poor(<50%)
2022-11-29,30.5,745.0,0.6669,0.4452,0.4373,0.0788,745.0,1.0,good(>=90%)
2022-12-14,19.0,745.0,0.6218,0.4221,0.4178,0.0541,745.0,1.0,good(>=90%)
2022-12-19,6.0,745.0,0.5893,0.3989,0.3898,0.0186,745.0,1.0,good(>=90%)
2022-12-24,39.0,745.0,0.5679,0.3862,0.3816,0.0617,745.0,1.0,good(>=90%)
2022-12-29,38.0,745.0,0.5853,0.4045,0.3922,0.0138,745.0,1.0,good(>=90%)
2023-01-03,25.0,745.0,0.5783,0.3705,0.3671,-0.0155,745.0,1.0,good(>=90%)
2023-01-08,93.0,191.0,0.3936,0.4719,0.258,0.0369,745.0,0.256,poor(<50%)
2023-01-13,51.0,258.0,0.5283,0.3989,0.3557,0.0519,745.0,0.346,poor(<50%)
2023-01-18,31.0,745.0,0.5801,0.4033,0.3962,0.0059,745.0,1.0,good(>=90%)
2023-01-23,9.0,745.0,0.5382,0.3418,0.3485,-0.054,745.0,1.0,good(>=90%)
2023-01-28,40.0,745.0,0.5279,0.3657,0.3469,-0.0421,745.0,1.0,good(>=90%)
2023-02-02,32.0,6.0,0.437,0.3275,0.2787,-0.0567,745.0,0.008,poor(<50%)
2023-02-07,50.0,165.0,0.5407,0.4806,0.3574,0.0684,745.0,0.221,poor(<50%)
2023-02-17,1.0,745.0,0.5765,0.4223,0.3898,0.0028,745.0,1.0,good(>=90%)
2023-02-22,11.0,745.0,0.5503,0.4055,0.3664,0.0141,745.0,1.0,good(>=90%)
2023-02-27,1.0,745.0,0.5616,0.4259,0.3797,0.0122,745.0,1.0,good(>=90%)
2023-03-09,16.0,654.0,0.4623,0.4392,0.3137,-0.0177,745.0,0.878,partial
2023-03-14,13.0,745.0,0.4979,0.3559,0.3261,-0.0735,745.0,1.0,good(>=90%)
2023-03-24,14.0,52.0,0.4427,0.418,0.2805,-0.0039,745.0,0.07,poor(<50%)
2023-04-08,78.0,745.0,0.5679,0.4494,0.3736,0.0045,745.0,1.0,good(>=90%)
2023-04-28,42.0,745.0,0.5301,0.5014,0.3527,0.0203,745.0,1.0,good(>=90%)
2023-05-13,39.0,745.0,0.6046,0.4743,0.3854,0.0572,745.0,1.0,good(>=90%)
2023-06-02,31.0,745.0,0.6958,0.6141,0.4533,0.1637,745.0,1.0,good(>=90%)
2023-06-22,5.0,745.0,0.7937,0.6571,0.5093,0.2521,745.0,1.0,good(>=90%)
2023-08-16,56.0,745.0,0.6041,0.5668,0.3908,0.1307,745.0,1.0,good(>=90%)
2023-09-20,76.0,745.0,0.6623,0.5777,0.4197,0.1309,745.0,1.0,good(>=90%)
2023-09-25,92.0,745.0,0.583,0.5434,0.3812,0.127,745.0,1.0,good(>=90%)
2023-10-15,82.0,745.0,0.6697,0.479,0.4513,0.1669,745.0,1.0,good(>=90%)
2023-10-20,42.0,745.0,0.7157,0.5322,0.4703,0.1623,745.0,1.0,good(>=90%)
2023-10-30,73.0,480.0,0.6353,0.5601,0.4303,0.135,745.0,0.644,partial
2023-11-09,94.0,362.0,0.5854,0.5852,0.4038,0.1952,745.0,0.486,poor(<50%)
2023-11-24,30.0,745.0,0.6732,0.4821,0.4622,0.1077,745.0,1.0,good(>=90%)
2023-11-29,0.0,745.0,0.6668,0.4559,0.4535,0.0842,745.0,1.0,good(>=90%)
2023-12-09,11.0,745.0,0.6814,0.4543,0.4704,0.0855,745.0,1.0,good(>=90%)
2023-12-19,38.0,745.0,0.6632,0.4425,0.4521,0.0718,745.0,1.0,good(>=90%)
2024-01-03,1.0,745.0,0.6368,0.4313,0.4465,0.0646,745.0,1.0,good(>=90%)
2024-01-08,29.0,745.0,0.6125,0.4158,0.4174,0.0506,745.0,1.0,good(>=90%)
2024-01-18,1.0,745.0,0.5399,0.3704,0.3695,-0.0143,745.0,1.0,good(>=90%)
2024-01-28,13.0,745.0,0.5231,0.3527,0.3527,-0.0383,745.0,1.0,good(>=90%)
2024-02-12,3.0,745.0,0.5735,0.4275,0.4064,0.0247,745.0,1.0,good(>=90%)
2024-02-17,51.0,408.0,0.4958,0.496,0.3428,0.0768,745.0,0.548,partial
2024-02-27,9.0,363.0,0.5293,0.4476,0.3624,0.006,745.0,0.487,poor(<50%)
2024-03-03,27.0,743.0,0.5264,0.5115,0.3686,0.0066,745.0,0.997,good(>=90%)
2024-03-08,29.0,745.0,0.4913,0.4753,0.3382,0.0089,745.0,1.0,good(>=90%)
2024-03-18,13.0,745.0,0.523,0.5833,0.366,0.0448,745.0,1.0,good(>=90%)
2024-04-02,11.0,133.0,0.5736,0.4926,0.396,-0.0114,745.0,0.179,poor(<50%)
2024-04-07,23.0,745.0,0.4696,0.5021,0.3167,0.0039,745.0,1.0,good(>=90%)
2024-04-22,30.0,745.0,0.5119,0.4522,0.3422,-0.015,745.0,1.0,good(>=90%)
2024-04-27,7.0,745.0,0.4947,0.4116,0.3203,-0.0353,745.0,1.0,good(>=90%)
2024-05-12,26.0,745.0,0.5325,0.419,0.3502,-0.0364,745.0,1.0,good(>=90%)
2024-05-27,30.0,741.0,0.5531,0.4365,0.3476,-0.0109,745.0,0.995,good(>=90%)
2024-08-05,42.0,745.0,0.6153,0.4846,0.3744,0.0491,745.0,1.0,good(>=90%)
2024-08-10,36.0,586.0,0.5844,0.6279,0.3792,0.1014,745.0,0.787,partial
2024-09-19,70.0,584.0,0.6809,0.6769,0.4578,0.1981,745.0,0.784,partial
2024-10-24,9.0,745.0,0.778,0.6215,0.5376,0.2352,745.0,1.0,good(>=90%)
2024-10-29,35.0,745.0,0.7033,0.5013,0.4772,0.1429,745.0,1.0,good(>=90%)
2024-11-03,30.0,745.0,0.6853,0.4892,0.4573,0.1092,745.0,1.0,good(>=90%)
2024-11-08,70.0,745.0,0.6789,0.4989,0.4634,0.1076,745.0,1.0,good(>=90%)
2024-11-18,38.5,745.0,0.6959,0.5152,0.4925,0.1242,745.0,1.0,good(>=90%)
2024-11-28,30.0,745.0,0.7078,0.5229,0.5034,0.1614,745.0,1.0,good(>=90%)
2024-12-03,57.0,745.0,0.7025,0.5686,0.4989,0.1641,745.0,1.0,good(>=90%)
2024-12-18,1.0,745.0,0.6516,0.4486,0.4615,0.0812,745.0,1.0,good(>=90%)
2025-01-02,0.0,745.0,0.609,0.4037,0.4251,0.0377,745.0,1.0,good(>=90%)
2025-01-07,3.0,745.0,0.6116,0.432,0.44,0.0649,745.0,1.0,good(>=90%)
2025-02-06,9.0,745.0,0.558,0.4097,0.3852,0.0201,745.0,1.0,good(>=90%)
2025-02-11,15.0,745.0,0.5541,0.3986,0.3616,0.013,745.0,1.0,good(>=90%)
2025-02-16,29.0,562.0,0.4985,0.4909,0.3488,0.0054,745.0,0.754,partial
2025-02-26,87.0,745.0,0.5708,0.4683,0.388,0.0733,745.0,1.0,good(>=90%)
2025-03-03,40.0,30.0,0.5583,0.4553,0.3603,0.0046,745.0,0.04,poor(<50%)
2025-03-18,61.0,744.0,0.6049,0.5388,0.4222,0.0887,745.0,0.999,good(>=90%)
2025-03-23,0.0,745.0,0.6179,0.5123,0.4079,0.0681,745.0,1.0,good(>=90%)
2025-03-28,20.0,745.0,0.6139,0.5427,0.4273,0.058,745.0,1.0,good(>=90%)
2025-04-07,57.0,144.0,0.4661,0.5546,0.3301,0.1076,745.0,0.193,poor(<50%)
2025-05-04,35.0,32.0,0.7439,0.692,0.4792,0.213,745.0,0.043,poor(<50%)
2025-05-07,60.0,707.0,0.6195,0.6283,0.4038,0.1474,745.0,0.949,good(>=90%)
2025-06-06,73.0,745.0,0.7111,0.6611,0.4779,0.2276,745.0,1.0,good(>=90%)
2025-07-26,81.0,35.0,0.6021,0.7482,0.4204,0.2063,745.0,0.047,poor(<50%)
2025-10-19,72.0,745.0,0.7212,0.5176,0.4468,0.1918,745.0,1.0,good(>=90%)
2025-11-13,81.0,657.0,0.5942,0.5133,0.4287,0.2448,745.0,0.882,partial
2025-12-03,12.0,745.0,0.7329,0.5126,0.5145,0.1874,745.0,1.0,good(>=90%)
2025-12-08,16.0,745.0,0.7253,0.5385,0.4889,0.2027,745.0,1.0,good(>=90%)
2025-12-10,87.0,261.0,0.5494,0.5643,0.3707,0.1473,745.0,0.35,poor(<50%)
2025-12-13,35.0,745.0,0.7067,0.5938,0.5057,0.1924,745.0,1.0,good(>=90%)
2025-12-23,27.0,296.0,0.643,0.3883,0.4412,0.1001,745.0,0.397,poor(<50%)
2025-12-28,21.0,745.0,0.6875,0.4549,0.4519,0.117,745.0,1.0,good(>=90%)
2025-12-30,3.0,745.0,0.6873,0.4555,0.4854,0.1304,745.0,1.0,good(>=90%)
2026-01-02,3.0,745.0,0.673,0.4736,0.4827,0.1348,745.0,1.0,good(>=90%)
2026-01-07,3.0,745.0,0.6665,0.4485,0.4277,0.1158,745.0,1.0,good(>=90%)
2026-01-12,9.0,745.0,0.6435,0.4436,0.4411,0.1027,745.0,1.0,good(>=90%)
2026-01-17,11.0,745.0,0.6414,0.4527,0.4112,0.0819,745.0,1.0,good(>=90%)
2026-01-27,3.0,745.0,0.6462,0.446,0.4286,0.0941,745.0,1.0,good(>=90%)
2026-02-01,0.0,745.0,0.6377,0.4437,0.4524,0.08,745.0,1.0,good(>=90%)
2026-02-11,12.0,745.0,0.5975,0.4374,0.4241,0.0701,745.0,1.0,good(>=90%)
2026-02-16,17.0,486.0,0.6093,0.4352,0.3888,0.0709,745.0,0.652,partial
2026-02-21,2.0,745.0,0.6506,0.4923,0.4559,0.115,745.0,1.0,good(>=90%)
2026-02-26,60.0,258.0,0.5753,0.6971,0.3845,0.1548,745.0,0.346,poor(<50%)
2026-03-03,32.0,0.0,0.7802,0.7556,0.5273,0.2729,745.0,0.0,poor(<50%)
2026-03-08,18.0,680.0,0.6625,0.5567,0.428,0.1223,745.0,0.913,good(>=90%)
2026-03-13,33.0,745.0,0.6287,0.57,0.4399,0.1312,745.0,1.0,good(>=90%)
2026-03-28,29.0,28.0,0.614,0.715,0.417,0.1283,745.0,0.038,poor(<50%)
2026-03-30,35.0,617.0,0.5959,0.5545,0.4127,0.0943,745.0,0.828,partial
2026-04-02,5.0,582.0,0.5834,0.5532,0.406,0.0724,745.0,0.781,partial
2026-04-07,17.0,745.0,0.6149,0.5312,0.3926,0.0764,745.0,1.0,good(>=90%)
2026-04-12,9.0,14.0,0.5646,0.5768,0.3922,0.0791,745.0,0.019,poor(<50%)
2026-04-17,24.0,745.0,0.5693,0.487,0.3647,0.021,745.0,1.0,good(>=90%)
2026-04-19,33.0,745.0,0.5632,0.4619,0.3782,0.0069,745.0,1.0,good(>=90%)
2026-04-22,34.0,745.0,0.588,0.5339,0.3932,0.0414,745.0,1.0,good(>=90%)
2026-04-27,28.0,745.0,0.5953,0.4868,0.3791,0.0261,745.0,1.0,good(>=90%)
2026-06-26,53.0,483.0,0.6043,0.6527,0.3709,0.1498,745.0,0.648,partial
2026-07-21,67.0,745.0,0.7018,0.6134,0.4702,0.2025,745.0,1.0,good(>=90%)
2026-08-25,61.0,741.0,0.6053,0.7223,0.3866,0.22,745.0,0.995,good(>=90%)
2026-09-04,78.0,14.0,0.5957,0.6437,0.3886,0.2228,745.0,0.019,poor(<50%)'''
s2 = pd.read_csv(io.StringIO(S2_CSV), parse_dates=['date'])
g = s2[s2.clear_frac >= 0.9].reset_index(drop=True)
# temporal filter: drop scenes > 0.08 below the median of neighbours within +-30 days (residual haze)
flag = [ (len(nb := g[(abs((g.date - r.date).dt.days) <= 30) & (g.index != i)]) >= 2) and (r.NDVI < nb.NDVI.median() - 0.08) for i, r in g.iterrows() ]
obs = g[~pd.Series(flag)].set_index('date')
print(len(s2), 'scene-days,', len(g), 'with >= 90% clear pixels,', sum(flag), 'removed by temporal filter ->', len(obs))
obs[['NDVI','EVI','NDRE','NDMI']].describe().round(3)

## 3–4. Model: stress scalars, water balance and dual-timescale canopy
Soil thresholds (0–100 cm, SoilGrids 2.0 mean): θ_wp = 0.181, θ_fc = 0.336, θ_sat = 0.493 m³ m⁻³.

In [ ]:
"""Dual-timescale light-use-efficiency canopy model for an evergreen durian orchard.
Weather (ERA5) -> stress scalars -> GPP -> leaf pool L (fast) + crown cover c (slow) -> NDVI.
"""
import numpy as np, pandas as pd
from numba import njit

# ---------------- fixed values from literature ----------------
EPS = 1.268          # g C MJ-1, MOD17 BPLUT evergreen broadleaf forest
VPD_MIN, VPD_MAX = 0.80, 3.10   # kPa, MOD17 BPLUT EBF (daytime mean VPD)
T_OPT, SIG_T = 28.0, 5.59       # degC; sigma from f(40 degC)=0.1
K_EXT = 0.5                      # Beer-Lambert extinction, broadleaf
NDVI_FULL = 0.90
KC = 0.90                        # crop coefficient (assumed, evergreen fruit tree)
P_DEPL = 0.50                    # FAO-56 depletion fraction
ZR = 1000.0                      # root-zone depth (mm)
SOIL = dict(wp=0.181, fc=0.336, sat=0.493)   # SoilGrids, 0-100 cm
TAW = (SOIL["fc"] - SOIL["wp"]) * ZR          # mm


def f_T(T):
    return np.exp(-(T - T_OPT) ** 2 / (2 * SIG_T ** 2))


def f_VPD_ramp(v):
    return np.clip((VPD_MAX - v) / (VPD_MAX - VPD_MIN), 0, 1)


def f_VPD_gauss(v, vopt=1.4, sig=0.8):
    return np.exp(-(v - vopt) ** 2 / (2 * sig ** 2))


def f_SM_trapezoid(theta):
    wp, fc, sat = SOIL["wp"], SOIL["fc"], SOIL["sat"]
    c = fc - P_DEPL * (fc - wp)
    w = sat - 0.10
    return np.where(theta <= wp, 0, np.where(theta < c, (theta - wp) / (c - wp),
                    np.where(theta <= w, 1, np.clip((sat - theta) / (sat - w), 0, 1))))


@njit(cache=True)
def _bucket(P, ET0, phi, taw, p, kc):
    raw = p * taw
    Dr = 0.0
    Ks = np.empty(len(P))
    for i in range(len(P)):
        k = 1.0 if Dr <= raw else max(0.0, (taw - Dr) / ((1 - p) * taw))
        Ks[i] = k
        Dr = Dr - P[i] + k * kc * ET0[i]
        Dr = min(taw, max(0.0, Dr))
        if Dr > raw:
            Dr -= phi * (Dr - raw)
    return Ks


def bucket(P, ET0, phi=0.0):
    """FAO-56 root-zone water balance with latent irrigation fraction phi
    (phi=0 rainfed; phi=1 depletion restored to RAW whenever exceeded)."""
    return _bucket(np.asarray(P, float), np.asarray(ET0, float), float(phi), TAW, P_DEPL, KC)


@njit(cache=True)
def _canopy(PAR, stress, fS, L0, c0, alpha, b0, b1, r, cmax, dual, eps, kext, gbar):
    n = len(PAR)
    L = np.empty(n); c = np.empty(n); gpp = np.empty(n)
    L[0] = L0; c[0] = c0
    for i in range(n - 1):
        fpar = 1 - np.exp(-kext * L[i])
        gpp[i] = eps * fpar * PAR[i] * stress[i]
        beta = b0 + b1 * (1 - fS[i])
        L[i + 1] = max(1e-3, L[i] + alpha * gpp[i] - beta * L[i])
        if dual:
            c[i + 1] = min(0.999, c[i] + r * (gpp[i] / gbar) * c[i] * (1 - c[i] / cmax))
        else:
            c[i + 1] = c[i]
    gpp[n - 1] = gpp[n - 2]
    return L, c, gpp


def simulate(drv, th, variant="dual", vpd="ramp", sm="bucket"):
    """drv: dict of daily arrays PAR, T, VPD, SM, P, ET0. th: parameter dict."""
    PAR = np.asarray(drv["PAR"], float)
    fT = f_T(drv["T"])
    fV = f_VPD_ramp(drv["VPD"]) if vpd == "ramp" else f_VPD_gauss(drv["VPD"])
    if sm == "bucket":
        fS = bucket(drv["P"], drv["ET0"], th.get("phi", 0.0))
    elif sm == "era5":
        fS = f_SM_trapezoid(drv["SM"])
    else:
        fS = np.ones(len(PAR))
    fS = np.asarray(fS, float)
    stress = np.asarray(fT * fV * fS, float)
    gbar = EPS * np.mean(PAR * stress) * 0.5
    L, c, gpp = _canopy(PAR, stress, fS, th["L0"], th.get("c0", 1.0), th["alpha"], th["b0"],
                        th.get("b1", 0.0), th.get("r", 0.0), th.get("cmax", 1.0),
                        variant == "dual", EPS, K_EXT, gbar)
    fpar = 1 - np.exp(-K_EXT * L)
    if "uwet" in th:   # two-layer mixing: tree crowns (cover c) + inter-row understory
        g = pd.Series(bucket(drv["P"], drv["ET0"], 0.0)).ewm(alpha=1.0 / th["tau_u"]).mean().values
        U = th["bg"] + (th["uwet"] - th["bg"]) * g
        ndvi = c * (th["bg"] + (NDVI_FULL - th["bg"]) * fpar) + (1 - c) * U
    else:
        U = np.full(len(PAR), th["bg"])
        ndvi = th["bg"] + (NDVI_FULL - th["bg"]) * c * fpar
    return dict(ndvi=ndvi, L=L, c=c, fpar=fpar, U=U, gpp_crown=gpp, gpp_ground=gpp * c,
                fT=fT, fV=fV, fS=fS, stress=stress)


def nectec_logistic(drv, th):
    """Annual-crop chain (NECTEC style): Gaussian VPD, LAI logistic in cumulative GPP."""
    fT = f_T(drv["T"]); fV = f_VPD_gauss(drv["VPD"]); fS = f_SM_trapezoid(drv["SM"])
    gpp = EPS * drv["PAR"] * fT * fV * fS
    cg = np.cumsum(gpp)
    lai = th["Lmax"] / (1 + np.exp(-th["a"] * (cg - th["b"])))
    fpar = 1 - np.exp(-K_EXT * lai)
    return dict(ndvi=th["bg"] + (NDVI_FULL - th["bg"]) * fpar)


## 5. Calibration (literature values fixed)

In [ ]:
drv = dict(PAR=d.PAR_MJ_m2_d.values, T=d.T_daytime_C.values, VPD=d.VPD_daytime_kPa.values,
           SM=d.SM_rootzone_0_100cm.values, P=d.P_mm_d.values, ET0=d.ET0_mm_d.values)
idx = d.index.get_indexer(obs.index); y = obs.NDVI.values; years = obs.index.year.values
B = dict(alpha=(1e-4, 0.1), b0=(1e-3, 0.1), bg=(0.0, 0.6), L0=(0.1, 8), phi=(0, 1), r=(0, 0.005), c0=(0.2, 1.0))
# ERA5-Land soil moisture on its own wetness scale (5th-95th percentile), dry-side ramp
lo_, hi_ = np.percentile(drv['SM'], [5, 95]); w_ = np.clip((drv['SM'] - lo_) / (hi_ - lo_), 0, 1)
FS_ERA5 = np.clip(w_ / (1 - P_DEPL), 0, 1)
def simulate_given(th, fS):
    PAR = np.asarray(drv['PAR'], float); stress = np.asarray(f_T(drv['T']) * f_VPD_ramp(drv['VPD']) * fS, float)
    gbar = EPS * np.mean(PAR * stress) * 0.5
    L, c, gpp = _canopy(PAR, stress, np.asarray(fS, float), th['L0'], th['c0'], th['alpha'], th['b0'], 0.0, th['r'], 1.0, True, EPS, K_EXT, gbar)
    return dict(ndvi=th['bg'] + (NDVI_FULL - th['bg']) * c * (1 - np.exp(-K_EXT * L)))
SPECS = {
 'M3_dual_bucket':    (dict(variant='dual', vpd='ramp',  sm='bucket'), ['alpha','b0','bg','L0','phi','r','c0']),
 'M3g_dual_gaussVPD': (dict(variant='dual', vpd='gauss', sm='bucket'), ['alpha','b0','bg','L0','phi','r','c0']),
 'M3s_ERA5SM_ownscale': (dict(sm='given'),                            ['alpha','b0','bg','L0','r','c0']),
 'M3n_dual_nowater':  (dict(variant='dual', vpd='ramp',  sm='none'),   ['alpha','b0','bg','L0','r','c0']),
}
def th_of(nm, x): th = dict(zip(SPECS[nm][1], x)); th['cmax'] = 1.0; return th
def pred(nm, x):
    if SPECS[nm][0].get('sm') == 'given': return simulate_given(th_of(nm, x), FS_ERA5)['ndvi'][idx]
    return simulate(drv, th_of(nm, x), **SPECS[nm][0])['ndvi'][idx]
# B1: annual-crop chain re-initialised every year (logistic LAI in GPP accumulated since day d0)
DOY = d.index.dayofyear.values
def pred_b1(x):
    Lmax, a, b, bg, d0, phi = x
    gpp = EPS * drv['PAR'] * f_T(drv['T']) * f_VPD_ramp(drv['VPD']) * bucket(drv['P'], drv['ET0'], phi)
    cg = pd.Series(gpp).groupby(np.cumsum(DOY == int(round(d0)))).cumsum().values
    lai = Lmax / (1 + np.exp(-a * (cg - b)))
    return (bg + (NDVI_FULL - bg) * (1 - np.exp(-K_EXT * lai)))[idx]
B1_BOUNDS = [(0.1, 8), (1e-5, 0.05), (0, 2500), (0, 0.6), (1, 365), (0, 1)]
def fit_f(f, bounds, mask, seed=0):
    obj = lambda x: float(np.mean((f(x)[mask] - y[mask])**2))
    de = differential_evolution(obj, bounds, seed=seed, maxiter=300, popsize=20, tol=1e-8, polish=False)
    lo, hi = np.array(bounds).T
    r = least_squares(lambda x: f(x)[mask] - y[mask], np.clip(de.x, lo+1e-12, hi-1e-12), bounds=(lo, hi))
    return r.x if 2*r.cost/mask.sum() <= de.fun else de.x
def fit(nm, mask, seed=0): return fit_f(lambda x: pred(nm, x), [B[n] for n in SPECS[nm][1]], mask, seed)
def metrics(yt, yp):
    e = yp - yt
    return dict(RMSE=np.sqrt(np.mean(e**2)), MAE=np.mean(np.abs(e)), R2=1-np.sum(e**2)/np.sum((yt-yt.mean())**2))
x = fit('M3_dual_bucket', np.ones(len(y), bool), seed=42); TH = th_of('M3_dual_bucket', x)
out = simulate(drv, TH, **SPECS['M3_dual_bucket'][0])
print({k: round(v, 5) for k, v in TH.items()})
print('in-sample', {k: round(v, 4) for k, v in metrics(y, out['ndvi'][idx]).items()})
print('leaf turnover time 1/beta0 = %.1f days' % (1/TH['b0']))
print('identifiable composite psi = alpha*eps*k = %.5f' % (TH['alpha']*EPS*K_EXT))
print('crown cover c: %.3f -> %.3f' % (out['c'][0], out['c'][-1]))

### Leave-one-year-out comparison (≈ 10–15 min). Set `RUN_CV = False` to skip.

In [ ]:
RUN_CV = True
def harmonic_X(t):
    doy = 2*np.pi*t.dayofyear.values/365.25; tt = (t - pd.Timestamp('2020-01-01')).days.values/365.25
    return np.c_[np.ones(len(t)), tt, np.sin(doy), np.cos(doy), np.sin(2*doy), np.cos(2*doy)]
if RUN_CV:
    cv = {k: np.full(len(y), np.nan) for k in ['B0_harmonic_trend', 'B1_annual_crop_reinit'] + list(SPECS)}
    X = harmonic_X(obs.index)
    for yr in np.unique(years):
        tr, te = years != yr, years == yr
        b = np.linalg.lstsq(X[tr], y[tr], rcond=None)[0]; cv['B0_harmonic_trend'][te] = (X@b)[te]
        cv['B1_annual_crop_reinit'][te] = pred_b1(fit_f(pred_b1, B1_BOUNDS, tr, int(yr)))[te]
        for nm in SPECS:
            cv[nm][te] = pred(nm, fit(nm, tr, seed=int(yr)))[te]
        print('fold', yr, 'done')
    display(pd.DataFrame({k: metrics(y, v) for k, v in cv.items()}).T.round(4))

## 6. Figures

In [ ]:
BLUE, ORANGE, AQUA, INK = '#2a78d6', '#eb6834', '#1baf7a', '#0b0b0b'
days = d.index
fig, ax = plt.subplots(4, 1, figsize=(9, 10), sharex=True)
ax[0].plot(days, pd.Series(out['fV'], days).rolling(15, center=True).mean(), color=BLUE, label='f(VPD)')
ax[0].plot(days, pd.Series(out['fS'], days).rolling(15, center=True).mean(), color=ORANGE, label='f(SM), water balance')
ax[0].plot(days, pd.Series(out['fT'], days).rolling(15, center=True).mean(), color=AQUA, label='f(T)')
ax[0].set_ylabel('Scalar'); ax[0].legend(ncol=3, loc='lower left')
ax[1].scatter(obs.index, y, s=10, color=INK, label='Sentinel-2 NDVI')
ax[1].plot(days, out['ndvi'], color=BLUE, label='Chain (calibrated)'); ax[1].set_ylabel('NDVI'); ax[1].legend()
ax[2].plot(days, out['c'], color=BLUE); ax[2].set_ylabel('Crown cover c')
ax[3].plot(days, out['L'], color=AQUA); ax[3].set_ylabel('Leaf area L')
plt.tight_layout(); plt.show()
# model-free canopy-memory scan
t0 = (obs.index - obs.index[0]).days.values; A_ = np.c_[np.ones_like(t0), t0]
res = y - A_ @ np.linalg.lstsq(A_, y, rcond=None)[0]
pot = pd.Series(drv['PAR'] * out['stress'], days); taus = np.arange(1, 181)
rr = [np.corrcoef(pot.ewm(alpha=1/t).mean().reindex(obs.index).values, res)[0, 1] for t in taus]
plt.figure(figsize=(5, 3)); plt.plot(taus, rr, color=BLUE); plt.axvline(1/TH['b0'], color=ORANGE, ls='--')
plt.xlabel('memory time scale (days)'); plt.ylabel('corr. with detrended NDVI'); plt.show()
print('model-free peak at', taus[int(np.argmax(rr))], 'days')

## 7. Statistical observation model and inference
Observation model: $y_j = \mathrm{NDVI}(t_j;\theta) + \epsilon_j$, $\epsilon \sim N(0, \sigma^2 R(\ell))$, $R_{ij} = \exp(-|t_i - t_j|/\ell)$ (Ornstein–Uhlenbeck errors, irregular sampling).

Steps: (1) residual correlogram, (2) likelihood-ratio test of correlated vs independent errors, (3) feasible GLS, (4) profile likelihood 95% intervals and practical identifiability, (5) profile of the turnover time under correlated vs independent errors. Runtime ≈ 10–15 min.

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.stats import chi2
NAMES = ['alpha', 'b0', 'bg', 'L0', 'phi', 'r', 'c0']
LO = np.array([B[k][0] for k in NAMES]); HI = np.array([B[k][1] for k in NAMES])
t = (obs.index - obs.index[0]).days.values.astype(float); n = len(y)
Dt = np.abs(t[:, None] - t[None, :])
predM3 = lambda x: pred('M3_dual_bucket', x)

def m2ll_ou(ell, r):
    R = np.exp(-Dt / ell); Lc = np.linalg.cholesky(R + 1e-10 * np.eye(n)); w = np.linalg.solve(Lc, r)
    return n * np.log(w @ w / n) + 2 * np.sum(np.log(np.diag(Lc)))
def fit_ell(r):
    res = minimize_scalar(lambda le: m2ll_ou(np.exp(le), r), bounds=(np.log(0.5), np.log(365)), method='bounded')
    return float(np.exp(res.x)), float(res.fun)
def whitener(ell):
    Lc = np.linalg.cholesky(np.exp(-Dt / ell) + 1e-10 * np.eye(n)); return Lc, 2 * np.sum(np.log(np.diag(Lc)))
def gls_fit(xs, Lc, free=None):
    free = np.arange(7) if free is None else free
    def res(z):
        x = xs.copy(); x[free] = z; return np.linalg.solve(Lc, predM3(x) - y)
    r = least_squares(res, np.clip(xs[free], LO[free] + 1e-9, HI[free] - 1e-9), bounds=(LO[free], HI[free]), x_scale='jac', max_nfev=2000)
    x = xs.copy(); x[free] = r.x; return x, 2 * r.cost

# (1)-(2) residual correlation and LR test, starting from the OLS fit x
x_ols = np.array([TH[k] for k in NAMES]); r0 = y - predM3(x_ols)
for a, b in [(1, 15), (15, 30), (30, 60), (60, 120), (120, 240)]:
    ii, jj = np.where(np.triu((Dt > a) & (Dt <= b)))
    print(f'lag {a}-{b} d: residual corr = {np.corrcoef(r0[ii], r0[jj])[0, 1]:.2f}  ({len(ii)} pairs)')
ell0, m2 = fit_ell(r0); LR = n * np.log(r0 @ r0 / n) - m2
print(f'OU correlation length = {ell0:.1f} d, LR = {LR:.1f}, p = {0.5 * chi2.sf(LR, 1):.1e}')
# (3) feasible GLS
x, ell = x_ols.copy(), ell0
for it in range(3):
    Lc, _ = whitener(ell); x, _ = gls_fit(x, Lc); ell, _ = fit_ell(y - predM3(x))
Lc, logdet = whitener(ell); xg, rss = gls_fit(x, Lc)
rw = np.linalg.solve(Lc, y - predM3(xg)) / np.sqrt(rss / n)
print(f'GLS: ell = {ell:.1f} d, sigma = {np.sqrt(rss / n):.3f}, whitened lag-1 corr = {np.corrcoef(rw[:-1], rw[1:])[0, 1]:.2f}')
print({k: round(v, 5) for k, v in zip(NAMES, xg)})

In [ ]:
# (4)-(5) profile likelihood
def profile(Lc_, xopt, grid, ki):
    free = np.array([i for i in range(7) if i != ki]); _, rs0 = gls_fit(xopt.copy(), Lc_); base = n * np.log(rs0 / n)
    dev, xw = [], xopt.copy()
    for gv in grid:
        best = None
        for st in [np.r_[xopt[:ki], gv, xopt[ki + 1:]], np.r_[xw[:ki], gv, xw[ki + 1:]]]:
            xf, rs = gls_fit(st.copy(), Lc_, free)
            if best is None or rs < best[1]: best = (xf, rs)
        xw = best[0]; dev.append(n * np.log(best[1] / n) - base)
    dev = np.array(dev); return dev - min(0, dev.min())
crit = chi2.ppf(0.95, 1)
G = {'b0': np.geomspace(0.006, 0.1, 40), 'alpha': np.geomspace(0.002, 0.06, 30), 'phi': np.linspace(0, 1, 26),
     'r': np.linspace(0, 0.005, 26), 'c0': np.linspace(0.2, 1.0, 26), 'bg': np.linspace(0, 0.6, 25), 'L0': np.geomspace(0.1, 8, 25)}
rows, PROF = [], {}
for k, g in G.items():
    ki = NAMES.index(k); dv = profile(Lc, xg, g, ki); PROF[k] = dv; ins = g[dv <= crit]
    status = 'identifiable' if (dv[0] > crit and dv[-1] > crit) else ('bounded below only' if dv[-1] <= crit else 'bounded above only')
    rows.append(dict(param=k, estimate=xg[ki], ci_low=ins.min(), ci_high=ins.max(), status=status)); print(k, 'done')
tab = pd.DataFrame(rows).set_index('param')
tab.loc['tau=1/b0'] = [1 / xg[1], 1 / tab.loc['b0', 'ci_high'], 1 / tab.loc['b0', 'ci_low'], tab.loc['b0', 'status']]
tab.loc['psi=alpha*eps*k'] = [xg[0] * EPS * K_EXT, tab.loc['alpha', 'ci_low'] * EPS * K_EXT, tab.loc['alpha', 'ci_high'] * EPS * K_EXT, tab.loc['alpha', 'status']]
display(tab)
dev_iid = profile(np.eye(n), gls_fit(xg.copy(), np.eye(n))[0], G['b0'], 1)
ins = G['b0'][dev_iid <= crit]; print(f'tau 95% CI, independent errors: {1/ins.max():.1f}-{1/ins.min():.1f} d')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
h = np.linspace(0, 240, 200); mids = [8, 22.5, 45, 90, 180]
emp = [np.corrcoef(r0[np.where(np.triu((Dt > a) & (Dt <= b)))[0]], r0[np.where(np.triu((Dt > a) & (Dt <= b)))[1]])[0, 1]
       for a, b in [(1, 15), (15, 30), (30, 60), (60, 120), (120, 240)]]
ax[0].plot(h, np.exp(-h / ell), color=BLUE, label=f'OU, ell = {ell:.0f} d'); ax[0].scatter(mids, emp, color=INK, label='empirical')
ax[0].axhline(0, color='grey', lw=0.8); ax[0].set_xlabel('lag (days)'); ax[0].set_ylabel('residual correlation'); ax[0].legend()
tau = 1 / G['b0']; o = np.argsort(tau)
ax[1].plot(tau[o], PROF['b0'][o], color=BLUE, label='correlated errors (OU)')
ax[1].plot(tau[o], dev_iid[o], color=ORANGE, label='independent errors')
ax[1].axhline(crit, ls='--', color='grey'); ax[1].set_xlim(8, 170); ax[1].set_ylim(0, 14)
ax[1].set_xlabel('leaf turnover time 1/beta0 (days)'); ax[1].set_ylabel('profile deviance'); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Sensitivity to the temporal NDVI screening rule
The temporal filter removes a scene when its NDVI lies more than $\delta$ below the median of the other retained scenes within $\pm w$ days (at least two neighbours).
The default is $\delta = 0.08$, $w = 30$ d. This section repeats the leave-one-year-out comparison for $\delta \in \{0.06, 0.08, 0.10\}$, for $w \in \{20, 45\}$ d at $\delta = 0.08$, and without the filter.
It reports out-of-year R² for B0, B1, M3 and M3s and the leaf turnover time $1/\beta_0$ of M3 (≈ 6 runs × 10–15 min; set `RUN_SENS = False` to skip).

In [ ]:
RUN_SENS = True
def screen(delta, win):
    if delta is None: return np.zeros(len(g), bool)
    return np.array([(len(nb := g[(abs((g.date - r.date).dt.days) <= win) & (g.index != i)]) >= 2) and
                     (r.NDVI < nb.NDVI.median() - delta) for i, r in g.iterrows()])
SETTINGS = [(None, 30), (0.06, 30), (0.08, 30), (0.10, 30), (0.08, 20), (0.08, 45)]
_keep = (idx, y, years)            # restore the default data afterwards
rows = []
if RUN_SENS:
    for delta, win in SETTINGS:
        fl = screen(delta, win); o = g[~fl].set_index('date')
        idx = d.index.get_indexer(o.index); y = o.NDVI.values; years = o.index.year.values
        X = harmonic_X(o.index); cvs = {k: np.full(len(y), np.nan) for k in ['B0', 'B1', 'M3', 'M3s']}; taus = []
        for yr in np.unique(years):
            tr, te = years != yr, years == yr
            b = np.linalg.lstsq(X[tr], y[tr], rcond=None)[0]; cvs['B0'][te] = (X @ b)[te]
            cvs['B1'][te] = pred_b1(fit_f(pred_b1, B1_BOUNDS, tr, int(yr)))[te]
            xm = fit('M3_dual_bucket', tr, seed=int(yr)); cvs['M3'][te] = pred('M3_dual_bucket', xm)[te]
            taus.append(1 / th_of('M3_dual_bucket', xm)['b0'])
            cvs['M3s'][te] = pred('M3s_ERA5SM_ownscale', fit('M3s_ERA5SM_ownscale', tr, seed=int(yr)))[te]
        xf = fit('M3_dual_bucket', np.ones(len(y), bool), seed=42)
        rows.append(dict(delta='none' if delta is None else delta, window_d=win, removed=int(fl.sum()), n=len(y),
                         **{f'R2_{k}': metrics(y, v)['R2'] for k, v in cvs.items()},
                         RMSE_M3=metrics(y, cvs['M3'])['RMSE'],
                         tau_full_d=1 / th_of('M3_dual_bucket', xf)['b0'], tau_folds_d=f'{min(taus):.0f}-{max(taus):.0f}'))
        print('done', delta, win)
    idx, y, years = _keep
    display(pd.DataFrame(rows).round(3))

## Appendix — Google Earth Engine script used for Sentinel-2
```javascript
var poly = ee.Geometry.Polygon([[[102.2123655,12.6494732],[102.2128604,12.6492678],[102.2132364,12.6497048],[102.213768,12.6506003],[102.2143889,12.6519322],[102.2127727,12.652682],[102.212306,12.6523418],[102.2114791,12.6507442],[102.2116678,12.6503253],[102.2118234,12.6494747],[102.2123655,12.6494732]]]);
var csp = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED');
var s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(poly).filterDate('2020-01-01','2026-09-30').linkCollection(csp, ['cs_cdf']);
// per image: clear = cs_cdf>=0.6 and SCL not in {3,8,9,10,11}; NDVI, EVI, NDRE(B8,B5), NDMI(B8,B11); mean over polygon at 10 m
```
Data sources: ERA5 (Hersbach et al., 2020) and ERA5-Land (Muñoz-Sabater et al., 2021) through Open-Meteo; Sentinel-2 L2A; SoilGrids 2.0; MOD17 BPLUT (Running & Zhao, 2015); FAO-56 (Allen et al., 1998).